In [ ]:
######################## P0 MOBILE-ION DENSITY SWEEP ########################

import os
import time
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

# -----------------------------------------------------------------------------
# Working directory
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# Execution switches
# -----------------------------------------------------------------------------
# Set all three to True for a complete new sweep.
#
# To only regenerate the comparison outputs from saved results, use:
RUN_PREBIAS = False
RUN_FD      = False
RUN_COMPARE = True
# to regenerate only comparison outputs.
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# Notebook filenames
# -----------------------------------------------------------------------------
PINN_NOTEBOOK = "CA_PINN_Solver.ipynb"
FD_NOTEBOOK   = "FD_Solver.ipynb"

# -----------------------------------------------------------------------------
# Final validated model and training settings
# -----------------------------------------------------------------------------
BASE_PHI = 0.10

BASE_ACTIVATION    = "tanh"
BASE_HIDDEN_LAYERS = 4
BASE_NEURONS       = 40

BASE_WCONS = 100.0

BASE_NUM_EPOCH_ADAM  = 2000
BASE_NUM_EPOCH_LBFGS = 10000
BASE_RUN_LBFGS       = True

# Representative seed selected from the validation study
BASE_SEED = 2024

# -----------------------------------------------------------------------------
# P0 sweep
# -----------------------------------------------------------------------------
P0_list = [
    1.0e20,
    1.0e21,
    1.0e22,
    1.0e23,
    1.0e24,
]

# -----------------------------------------------------------------------------
# Output directory
# -----------------------------------------------------------------------------
BASE_DIR = os.path.join(
    "results",
    "P0_sweep",
)

os.makedirs(BASE_DIR, exist_ok=True)

summary_path = os.path.join(
    BASE_DIR,
    "P0_sweep_summary.csv",
)

# -----------------------------------------------------------------------------
# Preserve all previously completed sweep results.
#
# New P0 cases are appended/updated by their unique tag.
# Existing case rows are retained even when PINN/FD are run.
# -----------------------------------------------------------------------------
if os.path.exists(summary_path):
    rows = pd.read_csv(
        summary_path
    ).to_dict("records")
else:
    rows = []

# -----------------------------------------------------------------------------
# Variables that may remain from the preceding notebook execution
# -----------------------------------------------------------------------------
LEFTOVERS = [
    "prebias_model",
    "main_model",
    "model",
    "layers",
    "results",

    "X_0",
    "X_r",
    "X_a",
    "X_c",
    "u_a",
    "u_c",

    "X_r_pre",
    "X_0_pre",
    "X_aPre",
    "X_cPre",

    "lb",
    "ub",
    "lb_pre",
    "ub_pre",
    "lb_main",
    "ub_main",

    "u0_interp_dict",
    "u_0",
    "x_0",

    "X_test",
    "X_list",
    "T_list",

    "psi_pred",
    "n_pred",
    "I_pred",
]

# -----------------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------------
def require_files(output_directory, filenames):
    """Confirm that all required output files are present."""

    missing = [
        filename
        for filename in filenames
        if not os.path.exists(
            os.path.join(output_directory, filename)
        )
    ]

    if missing:
        raise FileNotFoundError(
            f"Missing files in {output_directory}: {missing}"
        )

def save_summary(summary_rows, output_path):
    """Write the sweep summary to CSV."""

    dataframe = pd.DataFrame(summary_rows)
    dataframe.to_csv(output_path, index=False)

# -----------------------------------------------------------------------------
# Run the complete sweep
# -----------------------------------------------------------------------------
for P0_val in P0_list:

    # Folder names use compact scientific notation:
    # 1e+19 becomes 1e19
    P0_string = f"{P0_val:.0e}".replace("+", "")

    tag = (
        f"P0_{P0_string}_"
        f"phi{BASE_PHI:.2f}_"
        f"{BASE_ACTIVATION}_"
        f"{BASE_HIDDEN_LAYERS}x{BASE_NEURONS}_"
        f"seed{BASE_SEED}"
    )

    OUTPUT_DIR = os.path.join(
        BASE_DIR,
        tag,
    )

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True,
    )

    print("\n" + "=" * 100)
    print(f"Running P0 sweep case : {tag}")
    print(f"P0                    : {P0_val:.3e} m^-3")
    print(f"phi_a = phi_c         : {BASE_PHI:.2f} eV")
    print(f"activation            : {BASE_ACTIVATION}")
    print(
        f"architecture          : "
        f"{BASE_HIDDEN_LAYERS} hidden layers x "
        f"{BASE_NEURONS} neurons"
    )
    print(f"W_cons                : {BASE_WCONS}")
    print(f"Adam epochs           : {BASE_NUM_EPOCH_ADAM}")
    print(f"L-BFGS iterations     : {BASE_NUM_EPOCH_LBFGS}")
    print(f"seed                   : {BASE_SEED}")
    print(f"output directory       : {OUTPUT_DIR}")
    print("=" * 100)

    # -------------------------------------------------------------------------
    # Clear the preceding TensorFlow graph and Python objects
    # -------------------------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    # Use the same seed for every P0 value so that P0 is the controlled change
    os.environ["PYTHONHASHSEED"] = str(BASE_SEED)

    random.seed(BASE_SEED)
    np.random.seed(BASE_SEED)
    tf.random.set_seed(BASE_SEED)

    g = globals()

    for name in LEFTOVERS:
        if name in g:
            del g[name]

    # -------------------------------------------------------------------------
    # Pass physical and numerical settings into the notebooks
    # -------------------------------------------------------------------------
    g["P0"] = float(P0_val)

    g["phia"] = float(BASE_PHI)
    g["phic"] = float(BASE_PHI)

    g["OUTPUT_DIR"] = OUTPUT_DIR

    g["ACTIVATION"] = BASE_ACTIVATION
    g["HIDDEN_LAYERS"] = int(BASE_HIDDEN_LAYERS)
    g["NEURONS"] = int(BASE_NEURONS)

    g["W_cons"] = float(BASE_WCONS)

    g["NUM_EPOCH_ADAM"] = int(BASE_NUM_EPOCH_ADAM)
    g["NUM_EPOCH_LBFGS"] = int(BASE_NUM_EPOCH_LBFGS)
    g["RUN_LBFGS"] = bool(BASE_RUN_LBFGS)

    g["SEED"] = int(BASE_SEED)
    g["seed"] = int(BASE_SEED)

    t_case_start = time.time()

    pinn_runtime_s = np.nan
    fd_runtime_s = np.nan

    try:

        # =====================================================================
        # Run the pre-bias PINN
        # =====================================================================
        if RUN_PREBIAS:

            print(f"\n[{tag}] Starting pre-bias PINN...")

            t0 = time.time()

            get_ipython().run_line_magic("run", f'-i "{PINN_NOTEBOOK}"')
            pinn_runtime_s = time.time() - t0

            print(
                f"[{tag}] Pre-bias PINN completed in "
                f"{pinn_runtime_s:.1f} s."
            )

        # ---------------------------------------------------------------------
        # Save scaling quantities computed by the PINN notebook
        # ---------------------------------------------------------------------

        # =====================================================================
        # Run the finite-difference reference solver
        # =====================================================================
        if RUN_FD:

            print(f"\n[{tag}] Starting FD reference solver...")

            t0 = time.time()

            get_ipython().run_line_magic("run", f'-i "{FD_NOTEBOOK}"')
            fd_runtime_s = time.time() - t0

            print(
                f"[{tag}] FD solver completed in "
                f"{fd_runtime_s:.1f} s."
            )

        # =====================================================================
        # Compare PINN and FD results
        # =====================================================================
        if RUN_COMPARE:

            required_output_files = [
                "pinn_psi.csv",
                "pinn_n.csv",
                "pinn_I.csv",
                "fd_psi.csv",
                "fd_n.csv",
                "fd_I.csv",
            ]

            require_files(
                OUTPUT_DIR,
                required_output_files,
            )

            from validation_utils import compare_and_plot

            print(f"\n[{tag}] Comparing PINN and FD solutions...")

            metrics = compare_and_plot(
                title_suffix=tag,
                base_dir=OUTPUT_DIR,
            )

            case_runtime_s = time.time() - t_case_start

            row = {
                # Network configuration
                "activation": BASE_ACTIVATION,
                "hidden_layers": int(BASE_HIDDEN_LAYERS),
                "neurons_per_layer": int(BASE_NEURONS),
                "W_cons": float(BASE_WCONS),
                "Adam_epochs": int(BASE_NUM_EPOCH_ADAM),
                "LBFGS_iterations": int(BASE_NUM_EPOCH_LBFGS),
                "seed": int(BASE_SEED),

                # Case identification
                "tag": tag,
                "output_directory": OUTPUT_DIR,

                # Relative L2 errors
                "relL2_psi": float(metrics["relL2_psi"]),
                "relL2_n": float(metrics["relL2_n"]),
                "relL2_I": float(metrics["relL2_I"]),

                # Ion-conservation quantities
                "Imean_PINN": float(metrics["I_mean_PINN"]),
                "Imean_FD": float(metrics["I_mean_FD"]),
                "Imean_rel_diff": float(
                    metrics["I_mean_rel_error"]
                ),

                # Runtime
                "PINN_runtime_s": float(pinn_runtime_s),
                "FD_runtime_s": float(fd_runtime_s),
                "case_runtime_s": float(case_runtime_s),

                # Completion state
                "status": "ok",
            }

            # Re-plot mode: keep values recorded during training (runtimes and case
            # metadata) that are not recomputed when RUN_PREBIAS and RUN_FD are False.
            previous = next((r for r in rows if r.get("tag") == tag), None)
            if previous is not None:
                for key, value in previous.items():
                    if key not in row or pd.isna(row[key]):
                        row[key] = value
                if not (RUN_PREBIAS or RUN_FD) and "case_runtime_s" in previous:
                    row["case_runtime_s"] = previous["case_runtime_s"]

            rows = [existing for existing in rows if existing.get("tag") != tag]
            rows.append(row)

            save_summary(
                rows,
                summary_path,
            )

            print(f"\n[{tag}] Case completed successfully.")
            print(
                f"[{tag}] Summary updated at:\n"
                f"{summary_path}"
            )

    except Exception as error:

        case_runtime_s = time.time() - t_case_start

        print("\n" + "!" * 100)
        print(f"[{tag}] CASE FAILED")
        print(f"Error type : {type(error).__name__}")
        print(f"Error      : {error}")
        print("!" * 100)

        failure_row = {
            # Network configuration
            "activation": BASE_ACTIVATION,
            "hidden_layers": int(BASE_HIDDEN_LAYERS),
            "neurons_per_layer": int(BASE_NEURONS),
            "W_cons": float(BASE_WCONS),
            "Adam_epochs": int(BASE_NUM_EPOCH_ADAM),
            "LBFGS_iterations": int(BASE_NUM_EPOCH_LBFGS),
            "seed": int(BASE_SEED),

            # Case identification
            "tag": tag,
            "output_directory": OUTPUT_DIR,

            # Relative L2 errors
            "relL2_psi": np.nan,
            "relL2_n": np.nan,
            "relL2_I": np.nan,

            # Ion-conservation quantities
            "Imean_PINN": np.nan,
            "Imean_FD": np.nan,
            "Imean_rel_diff": np.nan,

            # Runtime
            "PINN_runtime_s": float(pinn_runtime_s),
            "FD_runtime_s": float(fd_runtime_s),
            "case_runtime_s": float(case_runtime_s),

            # Completion state
            "status": (
                f"failed: {type(error).__name__}: {error}"
            ),
        }

        rows = [existing for existing in rows if existing.get("tag") != tag]
        rows.append(failure_row)

        save_summary(
            rows,
            summary_path,
        )

        continue

# -----------------------------------------------------------------------------
# Final completion message
# -----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("P0 SWEEP COMPLETED")
print(f"Cases attempted : {len(P0_list)}")
print(f"Output root     : {BASE_DIR}")
print(f"Summary file    : {summary_path}")
print("=" * 100)

In [ ]:
################################################################################
# Comparison of PINN and FD solutions for different mobile-ion
# densities P0
#
# Figure format:
#   - Different color for each P0
#   - Solid line  = FD
#   - Dashed line = PINN
#   - All P0 values plotted on the same axes
#
# Panels:
#   (a) Electrostatic potential psi(x)
#   (b) Electron density n(x)
#   (c) Mobile-ion density I(x)
################################################################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -----------------------------------------------------------------------------
# Representative P0 values to plot
# -----------------------------------------------------------------------------
P0_plot_list = [
    1e20,
    1e21,
    1e22,
    1e23,
    1e24,
]

# -----------------------------------------------------------------------------
# High-contrast colors for each P0
# -----------------------------------------------------------------------------
P0_colors = {
    1e20: "purple",
    1e21: "black",
    1e22: "red",
    1e23: "blue",
    1e24: "green",
}

# -----------------------------------------------------------------------------
# Labels used in the legend
# -----------------------------------------------------------------------------
P0_labels = {
    1e20: r"$P_0=10^{20}\ \mathrm{m}^{-3}$",
    1e21: r"$P_0=10^{21}\ \mathrm{m}^{-3}$",
    1e22: r"$P_0=10^{22}\ \mathrm{m}^{-3}$",
    1e23: r"$P_0=10^{23}\ \mathrm{m}^{-3}$",
    1e24: r"$P_0=10^{24}\ \mathrm{m}^{-3}$",
}

# -----------------------------------------------------------------------------
# Case-folder names (same as in the P0 runner above)
# -----------------------------------------------------------------------------
cases = []

for P0_value in P0_plot_list:

    P0_string = f"{P0_value:.0e}".replace("+", "")

    folder = (
        f"P0_{P0_string}_"
        f"phi{BASE_PHI:.2f}_"
        f"{BASE_ACTIVATION}_"
        f"{BASE_HIDDEN_LAYERS}x{BASE_NEURONS}_"
        f"seed{BASE_SEED}"
    )

    cases.append((P0_value, folder))

# -----------------------------------------------------------------------------
# Required result files
# -----------------------------------------------------------------------------
required_files = [
    "fd_psi.csv",
    "fd_n.csv",
    "fd_I.csv",
    "pinn_psi.csv",
    "pinn_n.csv",
    "pinn_I.csv",
]

# -----------------------------------------------------------------------------
# Check folders and files before plotting
# -----------------------------------------------------------------------------
for P0_value, folder in cases:

    outdir = os.path.join(BASE_DIR, folder)

    if not os.path.isdir(outdir):
        raise FileNotFoundError(
            f"Case folder not found for P0 = {P0_value:.0e} m^-3:\n"
            f"{outdir}"
        )

    missing_files = [
        filename
        for filename in required_files
        if not os.path.isfile(os.path.join(outdir, filename))
    ]

    if missing_files:
        raise FileNotFoundError(
            f"Missing files for P0 = {P0_value:.0e} m^-3 in:\n"
            f"{outdir}\n\n"
            f"Missing: {missing_files}"
        )

print("All selected P0 folders and CSV files were found.")

# -----------------------------------------------------------------------------
# Plot settings
# -----------------------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",

    # Larger text
    "font.size": 13,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,

    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,

    "lines.linewidth": 2.2,

    "savefig.dpi": 600,
    "savefig.bbox": "tight",
})

# -----------------------------------------------------------------------------
# Create 1 x 3 figure
#
# (a) psi
# (b) n
# (c) I
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(12.5, 4.1)
)

# -----------------------------------------------------------------------------
# Read and plot all P0 cases
# -----------------------------------------------------------------------------
for P0_value, folder in cases:

    outdir = os.path.join(BASE_DIR, folder)

    color = P0_colors[P0_value]

    # -------------------------------------------------------------------------
    # Read FD data
    # -------------------------------------------------------------------------
    fd_psi = pd.read_csv(os.path.join(outdir, "fd_psi.csv"))
    fd_n   = pd.read_csv(os.path.join(outdir, "fd_n.csv"))
    fd_I   = pd.read_csv(os.path.join(outdir, "fd_I.csv"))

    # -------------------------------------------------------------------------
    # Read PINN data
    # -------------------------------------------------------------------------
    pinn_psi = pd.read_csv(os.path.join(outdir, "pinn_psi.csv"))
    pinn_n   = pd.read_csv(os.path.join(outdir, "pinn_n.csv"))
    pinn_I   = pd.read_csv(os.path.join(outdir, "pinn_I.csv"))

    # =========================================================================
    # PANEL (a): Electrostatic potential
    # =========================================================================
    ax[0].plot(
        fd_psi["x_nm"],
        fd_psi["psi_V"],
        color=color,
        linestyle="-",
        linewidth=2.0
    )

    ax[0].plot(
        pinn_psi["x_nm"],
        pinn_psi["psi_V"],
        color=color,
        linestyle="--",
        linewidth=2.0
    )

    # =========================================================================
    # PANEL (b): Electron density
    # =========================================================================
    ax[1].semilogy(
        fd_n["x_nm"],
        fd_n["n_m3"],
        color=color,
        linestyle="-",
        linewidth=2.0
    )

    ax[1].semilogy(
        pinn_n["x_nm"],
        pinn_n["n_m3"],
        color=color,
        linestyle="--",
        linewidth=2.0
    )

    # =========================================================================
    # PANEL (c): Mobile-ion density
    # =========================================================================
    ax[2].semilogy(
        fd_I["x_nm"],
        fd_I["I_m3"],
        color=color,
        linestyle="-",
        linewidth=2.0
    )

    ax[2].semilogy(
        pinn_I["x_nm"],
        pinn_I["I_m3"],
        color=color,
        linestyle="--",
        linewidth=2.0
    )

# -----------------------------------------------------------------------------
# Axis labels
# -----------------------------------------------------------------------------
ax[0].set_xlabel("Position (nm)")
ax[1].set_xlabel("Position (nm)")
ax[2].set_xlabel("Position (nm)")

ax[0].set_ylabel(r"$\psi$ (V)")
ax[1].set_ylabel(r"$n$ (m$^{-3}$)")
ax[2].set_ylabel(r"$I$ (m$^{-3}$)")

# -----------------------------------------------------------------------------
# Panel labels
# -----------------------------------------------------------------------------
panel_labels = ["(a)", "(b)", "(c)"]

for a, label in zip(ax, panel_labels):
    a.text(
        0.02,
        1.03,
        label,
        transform=a.transAxes,
        ha="left",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        clip_on=False
    )

# -----------------------------------------------------------------------------
# Common x limits and grid
# -----------------------------------------------------------------------------
for a in ax:
    a.set_xlim(0, 500)
    a.grid(
        True,
        which="major",
        linewidth=0.5,
        alpha=0.25
    )

# Add minor logarithmic grid to density panels
for a in [ax[1], ax[2]]:
    a.grid(
        True,
        which="minor",
        linewidth=0.3,
        alpha=0.12
    )

# -----------------------------------------------------------------------------
# Common legend
#
# First section:
#     colors identify P0
#
# Second section:
#     line style identifies numerical method
# -----------------------------------------------------------------------------

# P0-color handles
p0_handles = [
    Line2D(
        [0],
        [0],
        color=P0_colors[P0],
        linewidth=2.2,
        linestyle="-",
        label=P0_labels[P0]
    )
    for P0 in P0_plot_list
]

# Method-line-style handles
method_handles = [
    Line2D(
        [0],
        [0],
        color="0.25",
        linewidth=2.2,
        linestyle="-",
        label="FD"
    ),
    Line2D(
        [0],
        [0],
        color="0.25",
        linewidth=2.2,
        linestyle="--",
        label="PINN"
    ),
]

# Combine legend handles
legend_handles = p0_handles + method_handles

# -----------------------------------------------------------------------------
# Common legend above all three panels
# -----------------------------------------------------------------------------
fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.985),
    ncol=7,
    frameon=False,
    handlelength=2.3,
    columnspacing=1.25,
    handletextpad=0.5
)

# -----------------------------------------------------------------------------
# Layout adjustment
# -----------------------------------------------------------------------------
fig.subplots_adjust(
    left=0.075,
    right=0.985,
    bottom=0.16,
    top=0.81,
    wspace=0.32
)

# -----------------------------------------------------------------------------
# Save figure
# -----------------------------------------------------------------------------
pdf_path = os.path.join(
    BASE_DIR,
    "effect_of_P0_profiles_overlay.pdf"
)

png_path = os.path.join(
    BASE_DIR,
    "effect_of_P0_profiles_overlay.png"
)

plt.savefig(
    pdf_path,
    format="pdf",
    bbox_inches="tight"
)

plt.savefig(
    png_path,
    format="png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("\nSaved publication figures:")
print(pdf_path)
print(png_path)